# CUDA 설치 확인

In [1]:
import torch
print(f"PyTorch 버전: {torch.__version__}")
print(f"CUDA 사용 가능 여부: {torch.cuda.is_available()}")
print(f"현재 GPU 이름: {torch.cuda.get_device_name(0)}")

PyTorch 버전: 2.5.1+cu121
CUDA 사용 가능 여부: True
현재 GPU 이름: NVIDIA GeForce GTX 1070


In [1]:
import os
import json
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.nn.utils.rnn import pad_sequence
from tqdm.notebook import tqdm
from sklearn.model_selection import KFold

# GPU 사용을 위한 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"현재 사용 중인 장치: {device}")

현재 사용 중인 장치: cuda


# 경로 리스트업

In [2]:
# 경로 설정
base_train_dir = "./Train/"
base_morph_dir = "./Train/morpheme/"

# 01부터 16까지 폴더 이름 생성
user_ids = [f"{i:02d}" for i in range(1, 17)] 

# 실제로 존재하는 폴더인지 확인 (직관적 확인용)
for uid in user_ids:
    p = os.path.join(base_train_dir, uid)
    exists = "존재" if os.path.exists(p) else "미존재"
    print(f"사용자 {uid} 데이터 폴더: {exists}")

사용자 01 데이터 폴더: 존재
사용자 02 데이터 폴더: 존재
사용자 03 데이터 폴더: 존재
사용자 04 데이터 폴더: 존재
사용자 05 데이터 폴더: 존재
사용자 06 데이터 폴더: 존재
사용자 07 데이터 폴더: 존재
사용자 08 데이터 폴더: 존재
사용자 09 데이터 폴더: 존재
사용자 10 데이터 폴더: 존재
사용자 11 데이터 폴더: 존재
사용자 12 데이터 폴더: 존재
사용자 13 데이터 폴더: 존재
사용자 14 데이터 폴더: 존재
사용자 15 데이터 폴더: 존재
사용자 16 데이터 폴더: 존재


# 데이터 로드

In [3]:
X_list = []
y_list = []
word_to_idx = {}
idx_counter = 0

# 설정: 일단 테스트를 위해 20개 단어만 뽑아봅시다
TARGET_WORD_COUNT = 100 

for uid in tqdm(user_ids, desc="16명 데이터 통합 중"):
    morph_path = os.path.join(base_morph_dir, uid)
    coord_path = os.path.join(base_train_dir, uid)
    
    # 해당 사용자의 정답 JSON 파일들 가져오기
    morph_files = glob.glob(os.path.join(morph_path, "*_morpheme.json"))
    
    for j_path in morph_files:
        with open(j_path, 'r', encoding='utf-8') as f:
            meta = json.load(f)
        
        # 1. 정보 추출
        word = meta['data'][0]['attributes'][0]['name'] # 정답 단어
        start_t = meta['data'][0]['start']             # 시작 시간
        end_t = meta['data'][0]['end']                 # 종료 시간
        folder_name = meta['metaData']['name'].replace('.mp4', '')
        
        target_coord_dir = os.path.join(coord_path, folder_name)
        if not os.path.exists(target_coord_dir): continue

        # 2. 단어 라벨링 (Target 개수 제한)
        if word not in word_to_idx:
            if len(word_to_idx) >= TARGET_WORD_COUNT: continue
            word_to_idx[word] = idx_counter
            idx_counter += 1
        
        # 3. 프레임 추출 (30fps 가정)
        json_frames = sorted(glob.glob(os.path.join(target_coord_dir, "*.json")))
        fps = 30
        selected_frames = json_frames[int(start_t*fps) : int(end_t*fps)]
        
        # 4. 프레임별 키포인트 가공 (직관적 정규화)
        sequence = []
        for frame_path in selected_frames:
            with open(frame_path, 'r') as f:
                f_data = json.load(f)
            
            if not f_data['people']:
                sequence.append(np.zeros(132))
                continue
            
            person = f_data['people']
            lh = np.array(person['hand_left_keypoints_2d'])  # 63개
            rh = np.array(person['hand_right_keypoints_2d']) # 63개
            pose = np.array(person['pose_keypoints_2d'])     # 75개
            
            # [정규화] 코(Nose, pose 0번) 좌표를 (0,0)으로 이동
            # 사용자가 화면 어디에 있든 동작의 상대적 궤적만 남깁니다.
            ref_x, ref_y = pose[0], pose[1]
            if ref_x != 0 and ref_y != 0:
                for i in range(0, 63, 3):
                    if lh[i] != 0: lh[i] -= ref_x     # x
                    if lh[i+1] != 0: lh[i+1] -= ref_y # y
                    if rh[i] != 0: rh[i] -= ref_x     # x
                    if rh[i+1] != 0: rh[i+1] -= ref_y # y
            
            # 최종 132차원 조합
            combined = np.concatenate([lh, rh, pose[:6]]) 
            sequence.append(combined)
        
        # 유효한 동작만 추가
        if len(sequence) > 5:
            X_list.append(torch.FloatTensor(np.array(sequence)))
            y_list.append(word_to_idx[word])

print(f"총 {len(X_list)}개의 샘플 로드 완료!")
print(f"학습할 단어 목록: {list(word_to_idx.keys())}")

16명 데이터 통합 중:   0%|          | 0/16 [00:00<?, ?it/s]

총 7810개의 샘플 로드 완료!
학습할 단어 목록: ['고민', '뻔뻔', '수어', '남아', '눈', '독신', '음료수', '발가락', '슬프다', '자극', '안타깝다', '어색하다', '여아', '외국인', '영아', '신사', '뉴질랜드', '나사렛대학교', '알아서', '장애인', '열아홉번째', '침착', '성실', '학교연혁', '싫어하다', '급하다', '필기시험', '병문안', '검사', '결승전', '낚시터', '낚시대', '당뇨병', '독서', '매표소', '면역', '감기', '배드민턴', '변비', '병명', '보건소', '불면증', '불행', '붕대', '사위', '설사', '성병', '방충', '소화제', '손녀', '손자', '수면제', '수집가', '여행지', '예식장', '올림픽경기', '회복', '첫번째', '운동경기', '입원', '재혼', '진단서', '축구장', '치료', '치료법', '친아들', '퇴원', '한약', '한약방', '빈혈', '화상', '가래떡', '고깃국', '고추', '고추가루', '사골', '배추국', '꽈베기', '벌꿀', '꿀물', '냄비', '찬물', '다과', '지방경찰청장', '된장찌게', '돼지고기', '두부', '딸기', '떡국', '라면', '막걸리', '무', '밥그릇', '밥솥', '보신탕', '부엌', '소불고기', '비빔밥', '사과', '사이다']


# 텐서 변환 및 최종 확인

In [4]:
# 모든 영상의 길이를 가장 긴 영상에 맞춰서 패딩(0 채우기)
X_final = pad_sequence(X_list, batch_first=True)
y_final = torch.LongTensor(y_list)

print("--- 최종 데이터 정보 ---")
print(f"X (샘플수, 프레임수, 특징수): {X_final.shape}")
print(f"y (라벨수): {y_final.shape}")
print(f"클래스(단어) 개수: {len(word_to_idx)}")

--- 최종 데이터 정보 ---
X (샘플수, 프레임수, 특징수): torch.Size([7810, 123, 132])
y (라벨수): torch.Size([7810])
클래스(단어) 개수: 100


# 모델 학습, 평가

In [5]:
class SignLanguageClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers):
        super(SignLanguageClassifier, self).__init__()
        # batch_first=True: (Batch, Seq, Feature) 순서로 데이터를 받음
        self.gru = nn.GRU(input_dim, hidden_dim, num_layers, batch_first=True, dropout=0.3)
        # 드롭아웃 추가로 과적합 방지
        self.fc = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, x):
        # x: (Batch, Seq_len, Input_dim)
        out, _ = self.gru(x)
        # GRU의 마지막 타임스텝(-1)의 출력값만 사용해 분류
        out = self.fc(out[:, -1, :])
        return out

In [7]:
# 하이퍼파라미터 설정
input_size = 132    # (왼손 21*3 + 오른손 21*3 + 포즈 2*3)
hidden_size = 128   # 데이터 다양성을 고려해 64 -> 128로 상향
output_dim = len(word_to_idx)
num_layers = 2
k_folds = 5
num_epochs = 80     # 너무 길면 과적합되므로 논문 권장치 수준으로 조정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset = TensorDataset(X_final, y_final)
kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)
fold_results = []

for fold, (train_ids, val_ids) in enumerate(kf.split(X_final)):
    print(f"\n--- Fold {fold + 1} / {k_folds} 시작 ---")
    
    # 데이터 분리 및 로더 생성
    train_subsampler = torch.utils.data.SubsetRandomSampler(train_ids)
    val_subsampler = torch.utils.data.SubsetRandomSampler(val_ids)
    
    train_loader = DataLoader(dataset, batch_size=16, sampler=train_subsampler)
    val_loader = DataLoader(dataset, batch_size=16, sampler=val_subsampler)
    
    # 모델 초기화
    model = SignLanguageClassifier(input_size, hidden_size, output_dim, num_layers).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.0005) # 초기 학습률 소폭 하향
    criterion = nn.CrossEntropyLoss()
    
    # 학습
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        # 10 에포크마다 진행 상황 출력
        if (epoch + 1) % 10 == 0:
            print(f"Fold {fold+1} | Epoch {epoch+1}/{num_epochs} | Loss: {total_loss/len(train_loader):.4f}")
    
    # 검증 (해당 Fold 성능 측정)
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = model(batch_X)
            _, predicted = torch.max(outputs.data, 1)
            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()
    
    acc = 100 * correct / total
    print(f"Fold {fold + 1} 최종 정확도: {acc:.2f}%")
    fold_results.append(acc)

print(f"\n✅ 최종 결과: 평균 검증 정확도 {np.mean(fold_results):.2f}% (+/- {np.std(fold_results):.2f})")


--- Fold 1 / 5 시작 ---
Fold 1 | Epoch 10/80 | Loss: 2.6495
Fold 1 | Epoch 20/80 | Loss: 1.5095
Fold 1 | Epoch 30/80 | Loss: 1.1192
Fold 1 | Epoch 40/80 | Loss: 0.8687
Fold 1 | Epoch 50/80 | Loss: 0.7140
Fold 1 | Epoch 60/80 | Loss: 0.5924
Fold 1 | Epoch 70/80 | Loss: 0.4427
Fold 1 | Epoch 80/80 | Loss: 0.4156
Fold 1 최종 정확도: 80.03%

--- Fold 2 / 5 시작 ---
Fold 2 | Epoch 10/80 | Loss: 2.3336
Fold 2 | Epoch 20/80 | Loss: 1.3128
Fold 2 | Epoch 30/80 | Loss: 0.8942
Fold 2 | Epoch 40/80 | Loss: 0.6843
Fold 2 | Epoch 50/80 | Loss: 0.5413
Fold 2 | Epoch 60/80 | Loss: 0.4708
Fold 2 | Epoch 70/80 | Loss: 0.4289
Fold 2 | Epoch 80/80 | Loss: 0.3976
Fold 2 최종 정확도: 79.39%

--- Fold 3 / 5 시작 ---
Fold 3 | Epoch 10/80 | Loss: 2.2783
Fold 3 | Epoch 20/80 | Loss: 1.3364
Fold 3 | Epoch 30/80 | Loss: 0.9680
Fold 3 | Epoch 40/80 | Loss: 0.7310
Fold 3 | Epoch 50/80 | Loss: 0.5152
Fold 3 | Epoch 60/80 | Loss: 0.4965
Fold 3 | Epoch 70/80 | Loss: 0.3886
Fold 3 | Epoch 80/80 | Loss: 0.3841
Fold 3 최종 정확도: 76.76%



# 모델 저장

In [8]:
# 1. 전체 데이터로 로더 생성 (쪼개지 않음)
full_loader = DataLoader(dataset, batch_size=16, shuffle=True)

# 2. 모델 새로 초기화
final_model = SignLanguageClassifier(input_size, hidden_size, output_dim, num_layers).to(device)
optimizer = optim.Adam(final_model.parameters(), lr=0.0005)
criterion = nn.CrossEntropyLoss()

# 3. 전체 데이터로 짧고 굵게 학습 (이미 성능 검증은 끝났으니 50~80 에포크 정도)
final_model.train()
for epoch in tqdm(range(80), desc="최종 모델 학습 중"):
    for batch_X, batch_y in full_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = final_model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

# 4. 진짜 최종 모델 저장
save_path = "sign_language_model_v1.pth"

# 모델 가중치와 단어 사전을 하나의 딕셔너리로 묶어서 저장
torch.save({
    'model_state_dict': model.state_dict(),
    'word_to_idx': word_to_idx,
    'hidden_size': hidden_size, # 나중에 모델 구조를 똑같이 만들기 위해 이런 정보도 넣으면 좋아요
    'num_layers': num_layers
}, save_path)

print(f"✅ 모델과 단어 사전이 {save_path}에 통합 저장되었습니다.")

최종 모델 학습 중:   0%|          | 0/80 [00:00<?, ?it/s]

✅ 모델과 단어 사전이 sign_language_model_v1.pth에 통합 저장되었습니다.


# 모델 불러오기

In [9]:
# 1. 파일 불러오기
save_path = "sign_language_model_v1.pth"
checkpoint = torch.load(save_path)

# 2. 저장된 정보 추출
loaded_word_to_idx = checkpoint['word_to_idx']
idx_to_word = {v: k for k, v in loaded_word_to_idx.items()} # 숫자를 다시 단어로
hidden_size = checkpoint['hidden_size']
num_layers = checkpoint['num_layers']
output_dim = len(loaded_word_to_idx)

# 3. 모델 초기화 및 가중치 로드
# 주의: SignLanguageClassifier 클래스가 위쪽 셀에 정의되어 있어야 합니다.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_test = SignLanguageClassifier(132, hidden_size, output_dim, num_layers).to(device)
model_test.load_state_dict(checkpoint['model_state_dict'])
model_test.eval() # 반드시 평가 모드로 설정 (Dropout 비활성화)

print(f"✅ 모델 로드 완료! (인식 가능 단어 수: {output_dim}개)")

✅ 모델 로드 완료! (인식 가능 단어 수: 100개)


/tmp/ipykernel_614/3780987966.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(save_path)


# 불러온 모델 테스트

In [10]:
import random

# 전체 데이터 중 무작위로 5개만 뽑아서 테스트
test_indices = random.sample(range(len(X_final)), 5)

print("\n--- 무작위 테스트 결과 ---")
with torch.no_grad():
    for idx in test_indices:
        # 데이터 준비
        input_data = X_final[idx].unsqueeze(0).to(device) # (1, Seq_len, 132)
        target_label = y_final[idx].item()
        
        # 모델 예측
        output = model_test(input_data)
        _, predicted = torch.max(output, 1)
        pred_idx = predicted.item()
        
        # 단어 변환
        pred_word = idx_to_word[pred_idx]
        true_word = idx_to_word[target_label]
        
        result = "⭕ 일치" if pred_word == true_word else "❌ 불일치"
        print(f"[{result}] 실제 정답: {true_word: <10} | 모델 예측: {pred_word}")


--- 무작위 테스트 결과 ---
[⭕ 일치] 실제 정답: 급하다        | 모델 예측: 급하다
[⭕ 일치] 실제 정답: 학교연혁       | 모델 예측: 학교연혁
[❌ 불일치] 실제 정답: 두부         | 모델 예측: 예식장
[⭕ 일치] 실제 정답: 병문안        | 모델 예측: 병문안
[⭕ 일치] 실제 정답: 밥그릇        | 모델 예측: 밥그릇
